# BTC 5m Scalping Signals — Colab runner

This notebook:
1. Clones the repo
2. Installs dependencies
3. Runs a single evaluation (sanity check)
4. Runs a loop that prints a signal every N seconds
5. Lets you download the log files

**Notes**
- Every code cell starts by `cd`-ing into the repo, so you can re-run them in any order and a Colab runtime restart won't break things.
- Colab disconnects after ~90 min idle and caps sessions at ~12h. Fine for testing; use a VPS for 24/7.
- The signals are decision support, not a trading bot. Paper-trade before acting on them.

## 1. Clone the repo (run this first)
Change `BRANCH` to `main` after the feature branch is merged.

In [ ]:
import os, shutil

REPO = "https://github.com/lhonhanleonard/IndicatorTest.git"
BRANCH = "claude/btc-scalping-signals-uxqfA"
REPO_DIR = "/content/IndicatorTest"

os.chdir("/content")
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --branch {BRANCH} --depth 1 {REPO} {REPO_DIR}
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls

## 2. Install dependencies

In [ ]:
import os
os.chdir("/content/IndicatorTest")
!pip install -q -r requirements.txt

## 3. (Optional) Tweak config
Skip this cell to use the defaults in `config.yaml`.

In [ ]:
import os, yaml, pathlib
os.chdir("/content/IndicatorTest")

cfg_path = pathlib.Path("config.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

# Examples — uncomment to change:
# cfg["loop_seconds"] = 30
# cfg["thresholds"]["enter"] = 1.2
# cfg["weights"]["news"] = 0.3

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. One-shot evaluation (sanity check)
Runs a single pass so you can see the component breakdown.

In [ ]:
import os
os.chdir("/content/IndicatorTest")
!python -m scalper.main --once

## 5. Run a live loop in the notebook

Each line streams straight into the cell output and is appended to `logs/signals.log` + `logs/signals.jsonl`.

**To stop:** click the stop button next to the cell, or press `I, I`.

In [ ]:
import os, sys, json, time
from dataclasses import asdict
from pathlib import Path

REPO_DIR = "/content/IndicatorTest"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scalper.main import load_config, evaluate_once, format_line
from scalper.data import MarketData
from scalper.news import NewsFeed

cfg = load_config(Path("config.yaml"))
spot_ids = cfg.get("exchanges") or [cfg.get("exchange", "kraken")]
futures_ids = cfg.get("futures_exchanges") or [cfg.get("futures_exchange", "bybit")]
market = MarketData(spot_ids, futures_ids, cfg.get("futures_symbol"))
news = NewsFeed(cfg["news"]["rss_url"], cfg["news"]["keywords"])

log_dir = Path("logs"); log_dir.mkdir(exist_ok=True)
text_path = log_dir / "signals.log"
jsonl_path = log_dir / "signals.jsonl"

print(f"symbol={cfg['symbol']} tf={cfg['timeframe']} loop={cfg['loop_seconds']}s\n")

try:
    while True:
        try:
            decision, ts = evaluate_once(cfg, market, news)
            line = format_line(decision, ts)
            print(line); sys.stdout.flush()
            with open(text_path, "a") as f:
                f.write(line + "\n")
            with open(jsonl_path, "a") as f:
                f.write(json.dumps({"ts": ts.isoformat(), **asdict(decision)}) + "\n")
        except Exception as e:
            print(f"[error] {e}"); sys.stdout.flush()
        time.sleep(cfg["loop_seconds"])
except KeyboardInterrupt:
    print("\nstopped.")


## 6. Inspect / download the logs

In [ ]:
import os
os.chdir("/content/IndicatorTest")
!tail -n 20 logs/signals.log

In [ ]:
import os
os.chdir("/content/IndicatorTest")
try:
    from google.colab import files
    files.download("logs/signals.jsonl")
except ImportError:
    print("Not in Colab — open logs/signals.jsonl directly.")

## Troubleshooting

**`ModuleNotFoundError: No module named 'scalper'`** — the notebook is not inside the repo directory. Run cell 1 (clone) once, then rerun the cell that failed. Every cell above already `cd`s into `/content/IndicatorTest` before running, so this should be rare after cell 1 has run once per session.

**`No such file or directory: 'config.yaml'`** — same cause, same fix.

**Runtime disconnected** — Colab's 90 min idle / 12 h hard cap. For 24/7 use a VPS.